In [ ]:
# ============================================
# CELL 1 — Mount & Install
# ============================================
from google.colab import drive
drive.mount('/content/drive')

!pip install tensorflow scikit-learn -q

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import joblib, os, json

DRIVE_PATH = '/content/drive/MyDrive/action_analysis/'
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ============================================
# CELL 2 — Load Data
# ============================================
X = np.load(DRIVE_PATH + 'X_landmarks.npy')
y = np.load(DRIVE_PATH + 'y_labels.npy', allow_pickle=True)

print(f"X shape: {X.shape}")  # (N, 30, 51)
print(f"Classes: {np.unique(y)}")
print(f"Samples per class:")
for c in np.unique(y):
    print(f"  {c}: {np.sum(y==c)}")

In [ ]:
# ============================================
# CELL 3 — Preprocess
# ============================================
# Normalize keypoints to 0-1 range
X = X / np.max(X, axis=(1,2), keepdims=True)
X = np.nan_to_num(X)  # handle any NaN values

# Encode labels
le = LabelEncoder()
y_enc = to_categorical(le.fit_transform(y))
joblib.dump(le, DRIVE_PATH + 'label_encoder.pkl')

# Save class names for inference
class_names = list(le.classes_)
with open(DRIVE_PATH + 'classes.json', 'w') as f:
    json.dump(class_names, f)
print(f"✅ Classes saved: {class_names}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=42
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

In [ ]:
# ============================================
# CELL 4 — Build Model
# ============================================
num_classes = y_enc.shape[1]

model = Sequential([
    LSTM(128, return_sequences=True,
         input_shape=(30, 51)),
    BatchNormalization(),
    Dropout(0.3),

    LSTM(64, return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),

    LSTM(32),
    BatchNormalization(),
    Dropout(0.2),

    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ============================================
# CELL 5 — Train
# ============================================
callbacks = [
    EarlyStopping(patience=15, restore_best_weights=True),
    ReduceLROnPlateau(patience=7, factor=0.5, min_lr=1e-6),
    # Auto save best model to Drive
    ModelCheckpoint(
        DRIVE_PATH + 'best_model.h5',
        save_best_only=True,
        monitor='val_accuracy'
    )
]

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=callbacks
)

print(f"✅ Best val accuracy: {max(history.history['val_accuracy']):.2%}")

In [ ]:
# ============================================
# CELL 6 — Evaluate & Plot
# ============================================
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].legend()
plt.savefig(DRIVE_PATH + 'training_curves.png')
plt.show()

# Confusion matrix
y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Confusion Matrix')
plt.savefig(DRIVE_PATH + 'confusion_matrix.png')
plt.show()

print(classification_report(y_true, y_pred,
      target_names=class_names))

In [ ]:
# ============================================
# CELL 7 — Convert to TFLite for fast inference
# ============================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(DRIVE_PATH + 'action_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ TFLite model saved!")